In [0]:
"""
AeroPulse Enterprise Lakehouse Platform.

Reusable Bronze ingestion utilities.

This module contains common functionality for reading
source deliveries, adding Bronze metadata, and writing
to Delta Bronze tables.
"""

from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def read_source_delivery(
    spark,
    source_delivery_path: str,
    file_format: str,
) -> DataFrame:
    """
    Read a source delivery using the specified file format.

    Parameters
    ----------
    spark:
        Active SparkSession.
    source_delivery_path:
        Path of the source delivery.
    file_format:
        Source file format.

    Returns
    -------
    DataFrame
        Source delivery DataFrame.
    """

    normalized_file_format = file_format.lower()

    if normalized_file_format == "csv":

        return (
            spark.read
            .option("header", "true")
            .csv(source_delivery_path)
        )

    elif normalized_file_format == "json":

        return (
            spark.read
            .json(source_delivery_path)
        )

    elif normalized_file_format == "parquet":

        return (
            spark.read
            .parquet(source_delivery_path)
        )

    else:

        raise ValueError(
            f"Unsupported file format: {file_format}"
        )


def add_bronze_metadata(
    source_dataframe: DataFrame,
    source_system: str,
    pipeline_run_id: str,
) -> DataFrame:
    """
    Add AeroPulse Bronze technical metadata columns.
    """

    return (
        source_dataframe
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_source_file_path",
            F.input_file_name()
        )
        .withColumn(
            "_source_system",
            F.lit(source_system)
        )
        .withColumn(
            "_pipeline_run_id",
            F.lit(pipeline_run_id)
        )
    )


def write_to_bronze(
    bronze_dataframe: DataFrame,
    bronze_table: str,
) -> None:
    """
    Write processed data to a Bronze Delta table.
    """

    (
        bronze_dataframe.write
        .format("delta")
        .mode("append")
        .saveAsTable(bronze_table)
    )